# MPP Converter Output Reader

Read CSV output files from Azure Blob Storage using the MPP converter settings in `.env`.

In [1]:
import os
from io import BytesIO
from pathlib import Path

import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv


REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists():
    REPO_ROOT = Path.cwd().parent

load_dotenv(REPO_ROOT / ".env")

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING_MPP_CONVERTER")
container_name = os.getenv("AZURE_STORAGE_CONTAINER_MPP_CONVERTER_OUT")

if not connection_string:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING_MPP_CONVERTER is not configured in .env")

blob_service = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service.get_container_client(container_name)

print(f"Connected to Azure Blob container: {container_name}")

Connected to Azure Blob container: mppoutputnew


In [2]:
blobs = list(container_client.list_blobs())
csv_blobs = [blob for blob in blobs if blob.name.lower().endswith(".csv")]

pd.DataFrame(
    [
        {
            "name": blob.name,
            "size_bytes": blob.size,
            "last_modified": blob.last_modified,
        }
        for blob in csv_blobs
    ]
)

,name,size_bytes,last_modified
0,Plymouth WP1 schedule final.csv,21409,2026-05-03 16:01:55+00:00
1,Sunderland LEVI.csv,22267,2026-05-03 14:09:57+00:00
2,Surrey WP11 Schedule (new).csv,26322,2026-05-03 16:03:59+00:00
3,Surrey WP12 Schedule1.csv,22244,2026-05-03 14:13:06+00:00
4,Warrington Project Plan.csv,39226,2026-05-03 14:29:03+00:00
5,__Project_Template_CK.csv,27293,2026-05-04 19:29:56+00:00


In [3]:
def read_csv_blob(blob_name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download a CSV blob from the output container and return it as a DataFrame."""
    blob_client = container_client.get_blob_client(blob_name)
    content = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(content), **read_csv_kwargs)


csv_dataframes = {blob.name: read_csv_blob(blob.name) for blob in csv_blobs}

print(f"Loaded {len(csv_dataframes)} CSV blob(s).")
list(csv_dataframes.keys())

Loaded 6 CSV blob(s).


['Plymouth WP1 schedule final.csv',
 'Sunderland LEVI.csv',
 'Surrey WP11 Schedule (new).csv',
 'Surrey WP12 Schedule1.csv',
 'Warrington Project Plan.csv',
 '__Project_Template_CK.csv']

In [39]:
import pandas as pd

pattern = r"\b(\d+)\.\s*Gate\s+(\d+)\b"

gate_rows_list = []

for blob, df in csv_dataframes.items():
    filtered = df[df["TaskName"].str.contains(pattern, na=False, regex=True)].copy()
    
    if filtered.empty:
        continue

    # Extract the number after "Gate" as GateNumber
    filtered["GateNumber"] = filtered["TaskName"].str.extract(pattern)[1]

    # Add WorkPackage
    filtered["WorkPackage"] = blob

    # Keep only required columns in the requested order
    filtered = filtered[["WorkPackage", "GateNumber", "StartDate", "FinishDate", "WeekOfYear"]]

    gate_rows_list.append(filtered)

all_gate_rows = pd.concat(gate_rows_list, ignore_index=True) if gate_rows_list else pd.DataFrame(
    columns=["WorkPackage", "GateNumber", "StartDate", "FinishDate", "WeekOfYear"]
)

C:\Users\CR814QE\AppData\Local\Temp\ipykernel_78868\2547334707.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df["TaskName"].str.contains(pattern, na=False, regex=True)].copy()
C:\Users\CR814QE\AppData\Local\Temp\ipykernel_78868\2547334707.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df["TaskName"].str.contains(pattern, na=False, regex=True)].copy()
C:\Users\CR814QE\AppData\Local\Temp\ipykernel_78868\2547334707.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df["TaskName"].str.contains(pattern, na=False, regex=True)].copy()
C:\Users\CR814QE\AppData\Local\Temp\ipykernel_78868\2547334707.py:8: UserWarning: This pattern is interpreted as a regular expression, and has ma

In [40]:
all_gate_rows

,WorkPackage,GateNumber,StartDate,FinishDate,WeekOfYear
0,Plymouth WP1 schedule final.csv,1,2025-10-31,2025-11-03,45
1,Plymouth WP1 schedule final.csv,2,2026-01-30,2026-01-30,5
2,Plymouth WP1 schedule final.csv,3,2026-03-09,2026-03-09,11
3,Plymouth WP1 schedule final.csv,4,2026-06-16,2026-06-16,25
4,Sunderland LEVI.csv,1,2025-08-13,2025-08-13,33
5,Sunderland LEVI.csv,2,2026-02-06,2026-02-06,6
6,Sunderland LEVI.csv,3,2026-02-25,2026-02-25,9
7,Sunderland LEVI.csv,4,2026-08-03,2026-08-03,32
8,Surrey WP11 Schedule (new).csv,1,2025-12-26,2025-12-26,52
9,Surrey WP11 Schedule (new).csv,2,2026-04-08,2026-04-08,15


In [44]:
all_gate_rows[all_gate_rows.FinishDate.str[:4]=='2026']
gate_summary = (
    all_gate_rows
    .assign(GateCol="Gate" + all_gate_rows["GateNumber"].astype(str))
    .pivot_table(
        index="WorkPackage",
        columns="GateCol",
        values="WeekOfYear",
        aggfunc="first"   # use first in case of duplicates
    )
    .reset_index()
)

# Optional: ensure columns appear in order
desired_cols = ["WorkPackage", "Gate1", "Gate2", "Gate3", "Gate4"]
for col in desired_cols[1:]:
    if col not in gate_summary.columns:
        gate_summary[col] = pd.NA

gate_summary = gate_summary[desired_cols]

In [45]:
gate_summary

GateCol,WorkPackage,Gate1,Gate2,Gate3,Gate4
0,Plymouth WP1 schedule final.csv,45,5,11,25
1,Sunderland LEVI.csv,33,6,9,32
2,Surrey WP11 Schedule (new).csv,52,15,21,35
3,Surrey WP12 Schedule1.csv,52,15,24,38
4,Warrington Project Plan.csv,3,19,23,46
5,__Project_Template_CK.csv,44,7,13,24
